In [1]:
import os

from dotenv import load_dotenv

load_dotenv("../.env")

True

Setup the embedding model

In [2]:
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

Connect to Redis Cloud

In [3]:
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.vector_stores.redis import RedisVectorStore
from redisvl.schema import IndexSchema

redis_conn_string = os.getenv("REDIS_URL")
schema = IndexSchema.from_dict(
    {
        "index": {"name": "blue_horizon", "prefix": "blue_horizon"},
        # customize fields that are indexed
        "fields": [
            # required fields for llamaindex
            {"type": "tag", "name": "id"},
            {"type": "tag", "name": "doc_id"},
            {"type": "text", "name": "text"},
            # custom vector field for bge-small-en-v1.5 embeddings
            {
                "type": "vector",
                "name": "vector",
                "attrs": {
                    "dims": 384,
                    "algorithm": "hnsw",
                    "distance_metric": "cosine",
                },
            },
        ],
    },
)
vector_store = RedisVectorStore(schema=schema, redis_url=redis_conn_string, overwrite=False)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

14:08:17 redisvl.index.index INFO   Index already exists, not overwriting.


In [4]:
index = VectorStoreIndex.from_vector_store(vector_store=vector_store, storage_context=storage_context)
retriever = index.as_retriever(similarity_top_k=4)

In [5]:
retriever.retrieve("What swimming options are there?")

[NodeWithScore(node=TextNode(id_='FAQ000009', embedding=None, metadata={'category': 'amenities', 'subcategory': 'business', 'keywords': 'pool, swimming, recreation', 'last_updated': '2024-10-15'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='Question:\nIs there a swimming pool?\n\nAnswer:\nYes, we have both indoor and outdoor pools open from 6:00 AM to 10:00 PM.', mimetype='text/plain', start_char_idx=None, end_char_idx=None, metadata_seperator='\n', text_template='{metadata_str}\n\n{content}'), score=0.6770186424260001),
 NodeWithScore(node=TextNode(id_='FAQ000010', embedding=None, metadata={'category': 'amenities', 'subcategory': 'recreation', 'keywords': 'spa, wellness, treatments', 'last_updated': '2024-01-07'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='Question:\nDo you off

In [12]:
from pprint import pprint

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

# 1. Initialize your Language Model (LLM)
llm = ChatOpenAI(model="gpt-5.1", temperature=0)

system_prompt = """You are an assistant who helps people find out information about a hotel.
Your sole job is to query the database for information about the hotel using the tool provided.
Provide the information returned from the tool that is relevant to the user's query in a
well-formatted manner. If you decide to include an item and it has a description, be
sure to include that description.

Do not offer to do anything specific for the user. After you have answered the user's
query, simply ask if there is any other information you can provide about the hotel.

Do not imply that your results are exhaustive.

Assume that prices are in dollars.

Do not mention that you are searching a database, but you may mention that you are or
have perfomed a search.

Do not provide any instructions to the user concerning the hotel that were not provided
to you.
"""

@tool(parse_docstring=True)
def query_hotel_info(query: str) -> list[dict]:
    """Provide information about the hotel in response to a passed-in query.

    The query should be concise and not ask for many details.
    Accesses an FAQ database, information about hotel amenities, and information about
    hotel services. Nothing else.

    Args:
        query (string): The query string

    Returns:
        list[dict]: The retrieved strings and their associated metadata
    """
    print(f"query={query}")
    retrieved_nodes = retriever.retrieve(query)
    pprint(retrieved_nodes)

    return [{"metadata": node.metadata, "text": node.text} for node in retrieved_nodes]

agent = create_agent(
    model=llm,
    tools=[query_hotel_info],
    system_prompt=system_prompt,
)

In [14]:
prompt = "What dining options are there?"
result = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
print(result["messages"][-1].content)

query=dining options
[NodeWithScore(node=TextNode(id_='FAQ000011', embedding=None, metadata={'category': 'services', 'subcategory': 'room service', 'keywords': 'room service, dining, food', 'last_updated': '2024-04-05'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='Question:\nIs room service available?\n\nAnswer:\nRoom service is available 24/7 with a full menu during restaurant hours and a limited menu overnight.', mimetype='text/plain', start_char_idx=None, end_char_idx=None, metadata_seperator='\n', text_template='{metadata_str}\n\n{content}'), score=0.746651649475),
 NodeWithScore(node=TextNode(id_='FAQ000007', embedding=None, metadata={'category': 'amenities', 'subcategory': 'dining', 'keywords': 'breakfast, dining, restaurant', 'last_updated': '2024-01-10'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', met

In [15]:
prompt = "Where can I swim?"
result = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
print(result["messages"][-1].content)

query=swimming options pool beach
[NodeWithScore(node=TextNode(id_='FAQ000009', embedding=None, metadata={'category': 'amenities', 'subcategory': 'business', 'keywords': 'pool, swimming, recreation', 'last_updated': '2024-10-15'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='Question:\nIs there a swimming pool?\n\nAnswer:\nYes, we have both indoor and outdoor pools open from 6:00 AM to 10:00 PM.', mimetype='text/plain', start_char_idx=None, end_char_idx=None, metadata_seperator='\n', text_template='{metadata_str}\n\n{content}'), score=0.728300690651),
 NodeWithScore(node=TextNode(id_='AM000013', embedding=None, metadata={'category': 'Fitness Services', 'price': 70, 'duration': 60, 'availability': '6:00-22:00', 'location': 'Spa & Wellness Center', 'booking_required': 'True', 'min_notice_hours': 24}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, meta